# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading, exploring, and preparing the [FAIR^2](https://doi.org/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library. All dataset objects are referenced by their Croissant `@id` identifiers for full reproducibility and schema traceability.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id` values, fields and column IDs. This is important for addressing all subsequent `mlcroissant` operations by unique identifiers.

In [ ]:
# List all record sets and their IDs
print("Available Record Sets:")

record_sets = []
if hasattr(metadata, 'record_set'):
    rs_list = metadata.record_set
    if not isinstance(rs_list, list):
        rs_list = [rs_list]
    for rs in rs_list:
        rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"- {rs_name} (@id='{rs_id}')")
        record_sets.append(rs_id)
else:
    print('No record sets found in this dataset.')

In [ ]:
# For demonstration: Print field and column ids for each record set
for rs_id in record_sets:
    print(f"\nRecord Set @id: {rs_id}")
    rec_set = dataset.record_set(rs_id)
    if hasattr(rec_set, 'field'):
        fields = rec_set.field
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            print(f"  Field: {field_name} (@id='{field_id}')")
            # If columns exist
            if hasattr(field, 'column'):
                cols = field.column
                if not isinstance(cols, list):
                    cols = [cols]
                for col in cols:
                    col_id = getattr(col, '@id', None)
                    col_name = getattr(col, 'name', None)
                    print(f"    Column: {col_name} (@id='{col_id}')")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame, referencing each by their `@id`.

> NOTE: For this dataset, the Croissant metadata lists zero entries in `recordSet`, so let's attempt to auto-discover possible record sets by inspecting metadata fields such as `distribution` and use the default record set if necessary.

In [ ]:
# Try to enumerate all record sets. If empty, try loading without specifying a record set.
dfs = {}

record_set_ids = record_sets if len(record_sets) > 0 else [None]

for rs_id in record_set_ids:
    try:
        print(f"\nLoading records for record set {rs_id if rs_id else '(default)'}...")
        records = list(dataset.records(record_set=rs_id) if rs_id else dataset.records())
        if records:
            df = pd.DataFrame(records)
            dfs[rs_id or 'default'] = df
            print(f"Loaded DataFrame for record set {rs_id or 'default'} with shape {df.shape}.")
        else:
            print('No records found.')
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# See the columns in one of the DataFrames
main_rs_id = list(dfs.keys())[0]  # Choose the first one for demonstration
print(f"\nColumns in DataFrame (record set '{main_rs_id}'):")
print(dfs[main_rs_id].columns.tolist())
dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply basic filtering, normalization, and grouping. Substitute your chosen numeric and grouping fields by their `@id`, as discovered in the overview.

In [ ]:
# EDA on the main DataFrame
df = dfs[main_rs_id]

# Inspect column names to pick appropriate numeric/group fields
print("Column names:", df.columns.tolist())

# Example: Suppose one field is '@id': 'http://senscience.ai/age_at_second_crc', and another is 'http://senscience.ai/sex'
# Substitute these with actual found @id fields in your case
numeric_field_id = None
group_field_id = None

# Try to auto-pick likely numeric fields by dtype and group fields by name
for col in df.columns:
    # Choose first numeric-looking column (int/float, excluding IDs)
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    # Choose first non-numeric column as possible group field
    if group_field_id is None and not pd.api.types.is_numeric_dtype(df[col]):
        group_field_id = col
if not numeric_field_id or not group_field_id:
    print("Couldn't automatically determine numeric/group fields – please review DataFrame columns above.")

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Filter for values greater than a threshold (e.g., >10)
threshold = 10
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field if it exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization

Visualize the distribution or group-wise summary statistics of the chosen numeric field (`@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group, if both fields available
if group_field_id and numeric_field_id:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and process a dataset with a Croissant schema using `mlcroissant`. All dataset entities were referenced by their `@id` as per FAIR & reproducibility principles. You can extend this notebook with additional analysis, modeling, or visualization using the flexible DataFrame produced.

Key steps:
- Metadata and record set enumeration by `@id`
- Loading records and mapping Croissant fields to DataFrame columns
- Filtering and normalizing numeric fields
- Grouped summaries and visualization

Proceed to domain-specific analysis or ML pipeline integration as needed.